# Task 3 — Run QAOA on the Portfolio Hamiltonian

**Goal:** use the validated cost Hamiltonian from Task 2 to run QAOA for the six-stock, choose-three portfolio problem.

This notebook uses a statevector simulator and classical optimizer. It is a local simulation; it does not require IBM Quantum hardware or an API key.

Before running, ensure `02_portfolio_hamiltonian.ipynb` passed all 64 energy checks. This notebook independently rebuilds the same Hamiltonian so it remains self-contained.

## 1. Imports and challenge data

In [1]:
import numpy as np
from itertools import product
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

rng = np.random.default_rng(2026)
n, K, q = 6, 3, 0.5
lam = 1.0
mu = rng.uniform(0.05, 0.25, n)
A = rng.normal(0, 0.1, (n, n))
sigma = A @ A.T

def portfolio_cost(x, lam=lam):
    x = np.asarray(x, dtype=int)
    return float(q * (x @ sigma @ x) - mu @ x + lam * (x.sum() - K) ** 2)

print("Stocks:", n, "| Select:", K, "| lambda:", lam)
print("Qiskit and SciPy imports succeeded.")

Stocks: 6 | Select: 3 | lambda: 1.0
Qiskit and SciPy imports succeeded.


## 2. Rebuild the cost Hamiltonian

We use the same binary expansion as Task 2 and retain the identity coefficient, so the energy includes the complete objective.

In [2]:
a = q * np.diag(sigma) - mu + lam * (1 - 2 * K)
b = {(i, j): 2 * q * sigma[i, j] + 2 * lam
     for i in range(n) for j in range(i + 1, n)}

pauli_terms = {}
def add_term(label, coeff):
    pauli_terms[label] = pauli_terms.get(label, 0.0) + float(coeff)

identity = "I" * n
add_term(identity, lam * K**2)

for i in range(n):
    zi = ["I"] * n
    zi[n - 1 - i] = "Z"
    add_term(identity, a[i] / 2)
    add_term("".join(zi), -a[i] / 2)

for (i, j), coeff in b.items():
    add_term(identity, coeff / 4)
    zi = ["I"] * n
    zi[n - 1 - i] = "Z"
    add_term("".join(zi), -coeff / 4)
    zj = ["I"] * n
    zj[n - 1 - j] = "Z"
    add_term("".join(zj), -coeff / 4)
    zij = ["I"] * n
    zij[n - 1 - i] = "Z"
    zij[n - 1 - j] = "Z"
    add_term("".join(zij), coeff / 4)

cost_hamiltonian = SparsePauliOp.from_list(
    [(label, coeff) for label, coeff in pauli_terms.items() if abs(coeff) > 1e-12]
).simplify()

print("Hamiltonian Pauli terms:", len(cost_hamiltonian))

Hamiltonian Pauli terms: 22


## 3. Define QAOA circuit and objective

For each layer, the circuit applies:
- a cost unitary built from the diagonal Hamiltonian;
- a mixer unitary made from `RX` rotations on each qubit.

Because the Hamiltonian contains only `I`, `Z`, and `ZZ` terms, the cost unitary can be implemented directly with `RZ` and `RZZ` gates. We evaluate the exact statevector expectation, so there is no shot noise in this first simulation.

In [3]:
# Extract the identity, single-Z, and ZZ coefficients.
pauli_coeffs = {}
for label, coeff in zip(cost_hamiltonian.paulis.to_labels(), cost_hamiltonian.coeffs):
    pauli_coeffs[label] = float(np.real(coeff))

def qaoa_state(params, p):
    gammas = params[:p]
    betas = params[p:]
    qc = QuantumCircuit(n)
    qc.h(range(n))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        # Apply exp(-i * gamma * H_cost), term by term.
        for label, coeff in pauli_coeffs.items():
            z_positions = [pos for pos, char in enumerate(label) if char == "Z"]
            if len(z_positions) == 1:
                qubit = n - 1 - z_positions[0]
                qc.rz(2 * gamma * coeff, qubit)
            elif len(z_positions) == 2:
                qubit_a = n - 1 - z_positions[0]
                qubit_b = n - 1 - z_positions[1]
                qc.rzz(2 * gamma * coeff, qubit_a, qubit_b)
            # Identity terms only add a global phase and can be omitted from the circuit.

        for qubit in range(n):
            qc.rx(2 * beta, qubit)

    return Statevector.from_instruction(qc)

def expected_cost(params, p):
    state = qaoa_state(params, p)
    return float(np.real(state.expectation_value(cost_hamiltonian)))

print("QAOA circuit and expectation functions are ready.")

QAOA circuit and expectation functions are ready.


## 4. Optimize a p = 1 QAOA circuit

This is a first, small run to verify the end-to-end workflow. QAOA optimization is non-convex, so different starting parameters can lead to different results. We use a fixed random seed for reproducibility.

In [4]:
p = 1
rng_opt = np.random.default_rng(42)
initial_params = np.concatenate([
    rng_opt.uniform(0, np.pi, p),      # gammas
    rng_opt.uniform(0, np.pi / 2, p),  # betas
])

result = minimize(
    fun=lambda params: expected_cost(params, p),
    x0=initial_params,
    method="COBYLA",
    options={"maxiter": 250, "rhobeg": 0.5}
)

print("Optimizer success flag:", result.success)
print("Optimizer message:", result.message)
print("Function evaluations:", result.nfev)
print("Best expected penalized cost:", round(float(result.fun), 6))
print("Optimized gamma/beta:", np.round(result.x, 5))

Optimizer success flag: True
Optimizer message: Return from COBYLA because the trust region radius reaches its lower bound.
Function evaluations: 34
Best expected penalized cost: 0.110713
Optimized gamma/beta: [2.74708 1.33623]


## 5. Sample the optimized state and decode portfolios

We sample the optimized statevector probabilities, then calculate the original objective and selected-stock count for each observed bitstring. The displayed bitstring is in stock order: the leftmost character is stock 0.

In [5]:
state = qaoa_state(result.x, p)
probabilities = state.probabilities()
rng_sample = np.random.default_rng(123)
shots = 4096
sampled_indices = rng_sample.choice(2**n, size=shots, p=probabilities)

counts = {}
for index in sampled_indices:
    bits = format(index, f"0{n}b")
    # Qiskit's displayed bitstring is qubit n-1 ... qubit 0.
    # Convert to stock order, where stock 0 is the leftmost displayed character.
    stock_bits = bits[::-1]
    counts[stock_bits] = counts.get(stock_bits, 0) + 1

sample_rows = []
for bits, count in counts.items():
    x = np.array([int(bit) for bit in bits], dtype=int)
    sample_rows.append({
        "bitstring": bits,
        "stocks_selected": int(x.sum()),
        "classical_cost": portfolio_cost(x),
        "count": count,
        "frequency": count / shots,
    })

sample_rows.sort(key=lambda row: (-row["count"], row["classical_cost"]))
print("Shots:", shots)
print("Distinct bitstrings observed:", len(sample_rows))
print("\nMost frequent sampled bitstrings:")
for row in sample_rows[:10]:
    print(
        f'{row["bitstring"]} | selected={row["stocks_selected"]} | '
        f'cost={row["classical_cost"]:.6f} | count={row["count"]} | '
        f'frequency={row["frequency"]:.3f}'
    )

Shots: 4096
Distinct bitstrings observed: 63

Most frequent sampled bitstrings:
011001 | selected=3 | cost=-0.503015 | count=153 | frequency=0.037
110001 | selected=3 | cost=-0.464474 | count=142 | frequency=0.035
010011 | selected=3 | cost=-0.459651 | count=140 | frequency=0.034
101001 | selected=3 | cost=-0.409479 | count=139 | frequency=0.034
100101 | selected=3 | cost=-0.382259 | count=132 | frequency=0.032
011100 | selected=3 | cost=-0.409839 | count=128 | frequency=0.031
010101 | selected=3 | cost=-0.478709 | count=126 | frequency=0.031
111001 | selected=4 | cost=0.406208 | count=124 | frequency=0.030
011010 | selected=3 | cost=-0.407299 | count=121 | frequency=0.030
111000 | selected=3 | cost=-0.393170 | count=118 | frequency=0.029


## 6. Compare against the exact classical benchmark

The exact benchmark enumerates all 20 portfolios that select exactly three stocks. This comparison is a reference point; a single p=1 QAOA run is not expected to always return the optimum.

In [6]:
valid_portfolios = []
for bits_tuple in product([0, 1], repeat=n):
    x = np.array(bits_tuple, dtype=int)
    if x.sum() == K:
        valid_portfolios.append((portfolio_cost(x), "".join(map(str, bits_tuple)), x))

valid_portfolios.sort(key=lambda row: row[0])
best_valid_cost, best_valid_bits, best_valid_x = valid_portfolios[0]

valid_sample_counts = sum(row["count"] for row in sample_rows if row["stocks_selected"] == K)
best_observed = min(sample_rows, key=lambda row: row["classical_cost"])

print("Exact best valid bitstring:", best_valid_bits)
print("Exact best valid cost:", round(best_valid_cost, 6))
print("Observed sample with lowest classical cost:", best_observed["bitstring"])
print("Observed sample's classical cost:", round(best_observed["classical_cost"], 6))
print(f"Samples selecting exactly {K}: {valid_sample_counts}/{shots} "
      f"({valid_sample_counts / shots:.1%})")
print("Note: lowest observed sample may be invalid if it selects a different number of stocks.")

Exact best valid bitstring: 011001
Exact best valid cost: -0.503015
Observed sample with lowest classical cost: 011001
Observed sample's classical cost: -0.503015
Samples selecting exactly 3: 2372/4096 (57.9%)
Note: lowest observed sample may be invalid if it selects a different number of stocks.


## Task 3 completion checklist

- [ ] Notebook runs from top to bottom with the `qiskit-track2` kernel.
- [ ] QAOA optimization returns a finite expected cost.
- [ ] Sampling prints observed bitstrings and their costs.
- [ ] Exact classical benchmark is printed for comparison.

This is an initial p=1 implementation. The next task will systematically compare depths p=1, 2, 3, and 4 and calculate the challenge's approximation ratio. Do not add those experiments to this notebook.